# Multimodal Cardiac Fusion — Small & Practical (Binary Tasks)
**Tasks:** binary structural disease (`normal` vs `disease`) and binary arrhythmia (`normal` vs `arrhythmia`)

**Architecture:**
- ECG branch: small 1D ResNet → 3 temporal tokens → latent vector
- Echo branch: pretrained ResNet18 (grayscale) → 3‑frame tokens → latent vector
- Fusion: concatenation of global ECG & echo vectors → two small MLP heads

**Training strategy:**
1. Single‑modality sanity checks (ECG → arrhythmia, Echo → structural)
2. Pretrain each encoder on its "natural" task
3. Fusion with frozen encoders
4. Partial unfreezing of last encoder blocks

**Why binary first?**  With only ~3000 studies the model needs strong signal. Binary classification is more robust and provides a solid foundation before attempting multi‑class.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 – Imports and Configuration
# ═══════════════════════════════════════════════════════════════════
import os, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve)

# ── Paths ───────────────────────────────────────────────────────────
# Adjust these if needed.  Using relative paths – the notebook
# assumes the cache/ folder is in the same directory.
CACHE_DIR  = "cache"
OUTPUT_DIR = "model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Data ────────────────────────────────────────────────────────────
ECG_LEADS   = 12
ECG_LEN     = 5000
ECHO_SIZE   = 224
ECHO_FRAMES = 3

# ── Tasks (binary) ──────────────────────────────────────────────────
# Coarse labels are 0‑6; we map:
#   structural : normal = index 2 → 0, everything else → 1
#   arrhythmia : normal = index 1 → 0, everything else → 1
NORMAL_STRUCT_IDX = 2
NORMAL_ARR_IDX    = 1
NUM_CLASSES_STRUCT = 2
NUM_CLASSES_ARR    = 2
STRUCT_CLASSES = ['normal', 'disease']
ARR_CLASSES    = ['normal', 'arrhythmia']

# ── Model hyperparameters ───────────────────────────────────────────
LATENT_DIM = 128          # common latent size for both branches
ECG_TOKENS = 3            # number of temporal tokens after pooling
ECHO_TOKENS = 3           # one token per frame (ED/Mid/ES)

# ── Training ────────────────────────────────────────────────────────
BATCH_SIZE   = 16
EPOCHS       = 30
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 8
SEED         = 42

# ── Reproducibility ─────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 – Dataset (binary labels)
# ═══════════════════════════════════════════════════════════════════

class CardiacDataset(Dataset):
    """
    Returns (ecg, echo, label_struct_bin, label_arr_bin)
    """
    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path, mmap_mode='r')
        self.ecg  = data['ecg']             # (N, 12, 5000) float32
        self.echo = data['echo']            # (N, 3, 224, 224) uint8
        # coarse multi-class labels (0‑6)
        self.lbl_s = data['label_struct_coarse'].astype(np.int64)
        self.lbl_a = data['label_arr_coarse'].astype(np.int64)
        self.augment = augment
        # ImageNet-style normalisation for grayscale echo
        self.echo_mean = 0.449
        self.echo_std  = 0.226

    def __len__(self):
        return len(self.lbl_s)

    def _augment_ecg(self, ecg):
        # Light augmentation: amplitude jitter, noise, shift, lead dropout
        ecg = ecg * np.random.uniform(0.9, 1.1)
        ecg = ecg + np.random.randn(*ecg.shape).astype(np.float32) * 0.02
        shift = np.random.randint(0, 500)
        ecg = np.concatenate([ecg[:, shift:],
                               np.zeros((ecg.shape[0], shift), dtype=np.float32)], axis=1)
        if np.random.rand() > 0.5:
            drop_idx = np.random.choice(12, np.random.randint(1, 3), replace=False)
            ecg[drop_idx] = 0
        return ecg

    def _augment_echo(self, echo):
        if np.random.rand() > 0.5:
            echo = echo[:, :, ::-1].copy()
        if np.random.rand() > 0.5:
            echo = np.clip(echo.astype(np.float32) * np.random.uniform(0.9, 1.1), 0, 255).astype(np.uint8)
        return echo

    def __getitem__(self, idx):
        ecg  = self.ecg[idx].copy().astype(np.float32)
        echo = self.echo[idx].copy()
        # Per-lead z-score normalisation
        ecg = np.clip(np.nan_to_num(ecg, nan=0.0, posinf=5.0, neginf=-5.0), -5.0, 5.0)
        ecg = (ecg - ecg.mean(axis=1, keepdims=True)) / (ecg.std(axis=1, keepdims=True) + 1e-6)

        if self.augment:
            ecg  = self._augment_ecg(ecg)
            echo = self._augment_echo(echo)

        ecg_t  = torch.from_numpy(ecg)
        echo_t = torch.from_numpy(echo.astype(np.float32) / 255.0)
        echo_t = (echo_t - self.echo_mean) / self.echo_std

        # Convert to binary
        s_bin = 0 if self.lbl_s[idx] == NORMAL_STRUCT_IDX else 1
        a_bin = 0 if self.lbl_a[idx] == NORMAL_ARR_IDX else 1
        return (ecg_t, echo_t,
                torch.tensor(s_bin, dtype=torch.long),
                torch.tensor(a_bin, dtype=torch.long))

# ── Helper for quick counting ──────────────────────────────────────
def print_counts(dataset, name):
    s_cnt = np.bincount(dataset.lbl_s, minlength=7)
    a_cnt = np.bincount(dataset.lbl_a, minlength=7)
    print(f"\n{name} set:")
    print(f"  Structural: {dict(zip(range(7), s_cnt))}")
    print(f"  Arrhythmia: {dict(zip(range(7), a_cnt))}")

train_ds = CardiacDataset(os.path.join(CACHE_DIR, 'train.npz'), augment=True)
val_ds   = CardiacDataset(os.path.join(CACHE_DIR, 'val.npz'),   augment=False)
test_ds  = CardiacDataset(os.path.join(CACHE_DIR, 'test.npz'),  augment=False)
print_counts(train_ds, 'Train')
print_counts(val_ds,   'Val')
print_counts(test_ds,  'Test')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 – DataLoaders with weighted sampling for structural task
# ═══════════════════════════════════════════════════════════════════

# Inverse-frequency oversampling on structural binary labels
s_bin_all = (train_ds.lbl_s != NORMAL_STRUCT_IDX).astype(int)  # 0=normal, 1=disease
class_counts = np.bincount(s_bin_all, minlength=2)
class_counts = np.where(class_counts == 0, 1.0, class_counts)
sample_weights = 1.0 / class_counts[s_bin_all]
sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
print(f"Train batches: {len(train_loader)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 – Model Blocks (ECG encoder, Echo encoder, Fusion, Heads)
# ═══════════════════════════════════════════════════════════════════

# ── 1D Residual Block ──────────────────────────────────────────────
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, 1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm1d(out_ch))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + self.shortcut(x))
        return out

# ── ECG Encoder ────────────────────────────────────────────────────
class ECGEncoder(nn.Module):
    """
    Input: (B, 12, 5000) → output (B, ECG_TOKENS, LATENT_DIM)
    """
    def __init__(self, latent_dim=LATENT_DIM, n_tokens=ECG_TOKENS):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(12, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3, stride=2, padding=1))
        self.layer1 = self._make_stage(64, 64,  2, stride=1)
        self.layer2 = self._make_stage(64, 128, 2, stride=2)
        self.layer3 = self._make_stage(128,256, 2, stride=2)
        self.pool   = nn.AdaptiveAvgPool1d(n_tokens)
        self.proj   = nn.Conv1d(256, latent_dim, 1)  # project to latent_dim

    @staticmethod
    def _make_stage(in_ch, out_ch, blocks, stride):
        layers = [ResidualBlock1D(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock1D(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)      # (B, 64, L/4)
        x = self.layer1(x)    # (B, 64, ~same)
        x = self.layer2(x)    # (B,128, L/8)
        x = self.layer3(x)    # (B,256, L/16)
        x = self.pool(x)      # (B,256, n_tokens)
        x = self.proj(x)      # (B, latent_dim, n_tokens)
        x = x.permute(0, 2, 1).contiguous()  # (B, n_tokens, latent_dim)
        return x

# ── Echo Encoder (ResNet18, 1‑channel input) ───────────────────────
class EchoEncoder(nn.Module):
    """
    Input: (B, 3, 224, 224) → output (B, ECHO_TOKENS, LATENT_DIM)
    Uses a shared ResNet18 backbone, one frame at a time.
    """
    def __init__(self, latent_dim=LATENT_DIM, n_tokens=ECHO_TOKENS):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Adapt first conv to 1 channel
        w = backbone.conv1.weight.data.mean(dim=1, keepdim=True)
        backbone.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        backbone.conv1.weight.data = w
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # remove FC
        # Adds positional encoding per frame
        self.pos_emb = nn.Parameter(torch.randn(1, n_tokens, latent_dim) * 0.02)
        self.proj = nn.Linear(512, latent_dim)  # ResNet18 output 512

    def forward(self, x):
        B, F, H, W = x.shape
        assert F == ECHO_TOKENS, f"Expected {ECHO_TOKENS} frames"
        x = x.view(B * F, 1, H, W)
        feats = self.backbone(x).flatten(1)  # (B*F, 512)
        feats = self.proj(feats)              # (B*F, latent_dim)
        feats = feats.view(B, F, -1)          # (B, F, latent_dim)
        feats = feats + self.pos_emb
        return feats

# ── Fusion + Heads ──────────────────────────────────────────────────
class CardiacFusionModel(nn.Module):
    """
    Encodes ECG and echo, pools each to a single vector, concatenates,
    and applies two independent MLP heads (structural, arrhythmia).
    """
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.ecg_enc  = ECGEncoder(latent_dim)
        self.echo_enc = EchoEncoder(latent_dim)
        fusion_in = latent_dim * 2
        # Small MLP heads
        self.head_struct = nn.Sequential(
            nn.Linear(fusion_in, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES_STRUCT))
        self.head_arr = nn.Sequential(
            nn.Linear(fusion_in, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES_ARR))

    def forward(self, ecg, echo):
        ecg_tok  = self.ecg_enc(ecg)       # (B, dim)
        echo_tok = self.echo_enc(echo)
        ecg_pool  = ecg_tok.mean(dim=1)    # global pool
        echo_pool = echo_tok.mean(dim=1)
        fused = torch.cat([ecg_pool, echo_pool], dim=-1)
        return self.head_struct(fused), self.head_arr(fused)

    def freeze_encoders(self):
        for p in self.ecg_enc.parameters():
            p.requires_grad = False
        for p in self.echo_enc.parameters():
            p.requires_grad = False

    def unfreeze_last_block(self):
        # Unfreeze the last residual stage of ECG and the last ResNet block of echo
        for p in self.ecg_enc.layer3.parameters():
            p.requires_grad = True
        # Echo: unfreeze layer4
        for p in self.echo_enc.backbone[7].parameters():
            p.requires_grad = True

print("Model blocks defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 – Single‑modality sanity‑check models (optional)
# ═══════════════════════════════════════════════════════════════════

class ECGOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ECGEncoder()
        self.head = nn.Sequential(
            nn.Linear(LATENT_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES_ARR))
    def forward(self, ecg):
        x = self.encoder(ecg).mean(dim=1)
        return self.head(x)

class EchoOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EchoEncoder()
        self.head = nn.Sequential(
            nn.Linear(LATENT_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES_STRUCT))
    def forward(self, echo):
        x = self.encoder(echo).mean(dim=1)
        return self.head(x)

print("Baseline models defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 – ECG Pretraining (Arrhythmia task)
# ═══════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = correct = total = 0
    for ecg, _, _, lbl_a in loader:
        ecg, lbl_a = ecg.to(device), lbl_a.to(device)
        optimizer.zero_grad()
        logits = model(ecg)
        loss = criterion(logits, lbl_a)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * ecg.size(0)
        correct += (logits.argmax(1) == lbl_a).sum().item()
        total += ecg.size(0)
    return total_loss / total, correct / total

def validate_ecg(model, loader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = total = 0
    with torch.no_grad():
        for ecg, _, _, lbl_a in loader:
            ecg, lbl_a = ecg.to(device), lbl_a.to(device)
            logits = model(ecg)
            loss = criterion(logits, lbl_a)
            total_loss += loss.item() * ecg.size(0)
            total += ecg.size(0)
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(lbl_a.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    return total_loss / total, f1_score(labels, preds, average='macro', zero_division=0)

print("ECG pretraining ...")
ecg_model = ECGOnlyModel().to(DEVICE)
optimizer = torch.optim.AdamW(ecg_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
best_f1 = 0.0
no_improve = 0
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(ecg_model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1 = validate_ecg(ecg_model, val_loader, criterion, DEVICE)
    scheduler.step()
    flag = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0
        torch.save(ecg_model.encoder.state_dict(), os.path.join(OUTPUT_DIR, 'ecg_encoder_pretrained.pt'))
        flag = ' *'
    else:
        no_improve += 1
    print(f"Ep {ep:02d} | tr_loss={tr_loss:.3f} tr_acc={tr_acc:.3f} | val_loss={val_loss:.3f} val_f1={val_f1:.3f}{flag}")
    if no_improve >= PATIENCE:
        print("Early stopping")
        break
print(f"Best validation arrhythmia F1: {best_f1:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 – Echo Pretraining (Structural task)
# ═══════════════════════════════════════════════════════════════════

def train_one_epoch_echo(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = correct = total = 0
    for _, echo, lbl_s, _ in loader:
        echo, lbl_s = echo.to(device), lbl_s.to(device)
        optimizer.zero_grad()
        logits = model(echo)
        loss = criterion(logits, lbl_s)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * echo.size(0)
        correct += (logits.argmax(1) == lbl_s).sum().item()
        total += echo.size(0)
    return total_loss / total, correct / total

def validate_echo(model, loader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = total = 0
    with torch.no_grad():
        for _, echo, lbl_s, _ in loader:
            echo, lbl_s = echo.to(device), lbl_s.to(device)
            logits = model(echo)
            loss = criterion(logits, lbl_s)
            total_loss += loss.item() * echo.size(0)
            total += echo.size(0)
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(lbl_s.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    return total_loss / total, f1_score(labels, preds, average='macro', zero_division=0)

print("Echo pretraining ...")
echo_model = EchoOnlyModel().to(DEVICE)
optimizer = torch.optim.AdamW(echo_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
best_f1 = 0.0
no_improve = 0
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch_echo(echo_model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1 = validate_echo(echo_model, val_loader, criterion, DEVICE)
    scheduler.step()
    flag = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0
        torch.save(echo_model.encoder.state_dict(), os.path.join(OUTPUT_DIR, 'echo_encoder_pretrained.pt'))
        flag = ' *'
    else:
        no_improve += 1
    print(f"Ep {ep:02d} | tr_loss={tr_loss:.3f} tr_acc={tr_acc:.3f} | val_loss={val_loss:.3f} val_f1={val_f1:.3f}{flag}")
    if no_improve >= PATIENCE:
        print("Early stopping")
        break
print(f"Best validation structural F1: {best_f1:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8 – Fusion Model Training
#   Stage A: load pretrained encoders, freeze them, train heads only
#   Stage B: unfreeze last encoder blocks, fine‑tune with lower LR
# ═══════════════════════════════════════════════════════════════════

def train_fusion_epoch(model, loader, opt, crit_s, crit_a, device, use_amp=False):
    model.train()
    total_loss = s_corr = a_corr = n = 0
    for ecg, echo, lbl_s, lbl_a in loader:
        ecg, echo = ecg.to(device), echo.to(device)
        lbl_s, lbl_a = lbl_s.to(device), lbl_a.to(device)
        opt.zero_grad()
        out_s, out_a = model(ecg, echo)
        loss = crit_s(out_s, lbl_s) + crit_a(out_a, lbl_a)
        loss.backward()
        opt.step()
        total_loss += loss.item() * ecg.size(0)
        s_corr += (out_s.argmax(1) == lbl_s).sum().item()
        a_corr += (out_a.argmax(1) == lbl_a).sum().item()
        n += ecg.size(0)
    return total_loss / n, s_corr / n, a_corr / n

def validate_fusion(model, loader, crit_s, crit_a, device):
    model.eval()
    total_loss = s_corr = a_corr = n = 0
    s_preds, a_preds, s_labels, a_labels = [], [], [], []
    with torch.no_grad():
        for ecg, echo, lbl_s, lbl_a in loader:
            ecg, echo = ecg.to(device), echo.to(device)
            lbl_s, lbl_a = lbl_s.to(device), lbl_a.to(device)
            out_s, out_a = model(ecg, echo)
            loss = crit_s(out_s, lbl_s) + crit_a(out_a, lbl_a)
            total_loss += loss.item() * ecg.size(0)
            s_corr += (out_s.argmax(1) == lbl_s).sum().item()
            a_corr += (out_a.argmax(1) == lbl_a).sum().item()
            n += ecg.size(0)
            s_preds.append(out_s.argmax(1).cpu())
            a_preds.append(out_a.argmax(1).cpu())
            s_labels.append(lbl_s.cpu())
            a_labels.append(lbl_a.cpu())
    s_preds  = torch.cat(s_preds)
    a_preds  = torch.cat(a_preds)
    s_labels = torch.cat(s_labels)
    a_labels = torch.cat(a_labels)
    s_f1 = f1_score(s_labels, s_preds, average='macro', zero_division=0)
    a_f1 = f1_score(a_labels, a_preds, average='macro', zero_division=0)
    return total_loss / n, s_corr / n, a_corr / n, s_f1, a_f1

fusion_model = CardiacFusionModel().to(DEVICE)
# Load pretrained encoders
fusion_model.ecg_enc.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, 'ecg_encoder_pretrained.pt'), map_location=DEVICE))
fusion_model.echo_enc.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, 'echo_encoder_pretrained.pt'), map_location=DEVICE))
print("Pretrained encoders loaded.")

crit_s = nn.CrossEntropyLoss()
crit_a = nn.CrossEntropyLoss()

# ── Stage A: freeze encoders, train fusion heads only ──────────────
fusion_model.freeze_encoders()
opt = torch.optim.AdamW(
    list(fusion_model.head_struct.parameters()) +
    list(fusion_model.head_arr.parameters()),
    lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-6)

best_f1 = 0.0
no_improve = 0
for ep in range(1, 11):
    tr_loss, tr_s, tr_a = train_fusion_epoch(fusion_model, train_loader, opt, crit_s, crit_a, DEVICE)
    vl, vs, va, vs_f1, va_f1 = validate_fusion(fusion_model, val_loader, crit_s, crit_a, DEVICE)
    scheduler.step()
    avg_f1 = (vs_f1 + va_f1) / 2
    flag = ''
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        no_improve = 0
        torch.save(fusion_model.state_dict(), os.path.join(OUTPUT_DIR, 'fusion_best.pt'))
        flag = ' *'
    else:
        no_improve += 1
    print(f"Ep {ep:02d} | tr {tr_loss:.3f} s={tr_s:.3f} a={tr_a:.3f} | "
          f"val {vl:.3f} s={vs:.3f} a={va:.3f} s_f1={vs_f1:.3f} a_f1={va_f1:.3f}{flag}")
    if no_improve >= PATIENCE:
        print("Early stopping")
        break

# ── Stage B: unfreeze last blocks, fine‑tune with lower LR ────────
print("\nPartial unfreezing ...")
fusion_model.unfreeze_last_block()
opt = torch.optim.AdamW([
    {'params': fusion_model.ecg_enc.parameters(), 'lr': 1e-5},
    {'params': fusion_model.echo_enc.parameters(), 'lr': 1e-5},
    {'params': list(fusion_model.head_struct.parameters()) +
               list(fusion_model.head_arr.parameters()), 'lr': 1e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-6)

no_improve = 0
for ep in range(1, 11):
    tr_loss, tr_s, tr_a = train_fusion_epoch(fusion_model, train_loader, opt, crit_s, crit_a, DEVICE)
    vl, vs, va, vs_f1, va_f1 = validate_fusion(fusion_model, val_loader, crit_s, crit_a, DEVICE)
    scheduler.step()
    avg_f1 = (vs_f1 + va_f1) / 2
    flag = ''
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        no_improve = 0
        torch.save(fusion_model.state_dict(), os.path.join(OUTPUT_DIR, 'fusion_best.pt'))
        flag = ' *'
    else:
        no_improve += 1
    print(f"Ep {ep:02d} | tr {tr_loss:.3f} s={tr_s:.3f} a={tr_a:.3f} | "
          f"val {vl:.3f} s={vs:.3f} a={va:.3f} s_f1={vs_f1:.3f} a_f1={va_f1:.3f}{flag}")
    if no_improve >= PATIENCE:
        print("Early stopping")
        break
print(f"Best validation avg F1: {best_f1:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 9 – Test Set Evaluation
# ═══════════════════════════════════════════════════════════════════

fusion_model.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, 'fusion_best.pt'), map_location=DEVICE))

def evaluate_test(model, loader, device):
    model.eval()
    s_preds, a_preds, s_labels, a_labels = [], [], [], []
    with torch.no_grad():
        for ecg, echo, lbl_s, lbl_a in loader:
            ecg, echo = ecg.to(device), echo.to(device)
            out_s, out_a = model(ecg, echo)
            s_preds.append(out_s.argmax(1).cpu())
            a_preds.append(out_a.argmax(1).cpu())
            s_labels.append(lbl_s.cpu())
            a_labels.append(lbl_a.cpu())
    s_preds  = torch.cat(s_preds)
    a_preds  = torch.cat(a_preds)
    s_labels = torch.cat(s_labels)
    a_labels = torch.cat(a_labels)
    return s_labels, s_preds, a_labels, a_preds

s_labels, s_preds, a_labels, a_preds = evaluate_test(fusion_model, test_loader, DEVICE)

def print_metrics(name, labels, preds, class_names):
    print(f"\n{name}:")
    print(classification_report(labels, preds, target_names=class_names, digits=3))
    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f"{name} Confusion Matrix")
    plt.tight_layout()
    plt.show()
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    print(f"Macro F1: {f1:.3f}")
    # AUC if binary (both classes present)
    if len(np.unique(labels)) == 2:
        # we need probabilities, so we temporarily get them
        # simple: just compute AUC from raw logits? We'll skip proper AUC for brevity
        pass

print_metrics("Structural Disease", s_labels, s_preds, STRUCT_CLASSES)
print_metrics("Arrhythmia", a_labels, a_preds, ARR_CLASSES)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 10 – Simple Interpretation (sample‑wise prediction table)
# ═══════════════════════════════════════════════════════════════════

def show_predictions(model, loader, device, n=5):
    model.eval()
    samples = []
    with torch.no_grad():
        for ecg, echo, lbl_s, lbl_a in loader:
            ecg, echo = ecg.to(device), echo.to(device)
            out_s, out_a = model(ecg, echo)
            pred_s = out_s.argmax(1).cpu()
            pred_a = out_a.argmax(1).cpu()
            for i in range(len(lbl_s)):
                samples.append((lbl_s[i].item(), lbl_a[i].item(),
                                pred_s[i].item(), pred_a[i].item()))
                if len(samples) >= n:
                    break
            if len(samples) >= n:
                break
    print(f"{'S true':>7} {'S pred':>7} {'A true':>7} {'A pred':>7}")
    for s_t, a_t, s_p, a_p in samples:
        print(f"{STRUCT_CLASSES[s_t]:>7} {STRUCT_CLASSES[s_p]:>7} {ARR_CLASSES[a_t]:>7} {ARR_CLASSES[a_p]:>7}")

show_predictions(fusion_model, test_loader, DEVICE, n=10)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Save final model
# ═══════════════════════════════════════════════════════════════════
final_path = os.path.join(OUTPUT_DIR, 'cardiac_fusion_small_final.pt')
torch.save({
    'model_state': fusion_model.state_dict(),
    'classes_struct': STRUCT_CLASSES,
    'classes_arr': ARR_CLASSES,
    'config': {
        'latent_dim': LATENT_DIM,
        'ecg_tokens': ECG_TOKENS,
        'echo_tokens': ECHO_TOKENS,
    },
}, final_path)
print(f"Model saved to {final_path}")